In [1]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ==========================================
# 0. DARK THEME CSS INJECTION
# ==========================================
display(HTML("""
<style>
    /* Dark theme for Jupyter Widgets */
    .jupyter-widgets {
        background-color: #1e1e1e !important;
        color: #ffffff !important;
    }
    .widget-label {
        color: #ffffff !important;
    }
    .widget-dropdown select {
        background-color: #333333 !important;
        color: #ffffff !important;
        border: 1px solid #555555 !important;
    }
    .widget-button {
        background-color: #333333 !important;
        color: #ffffff !important;
        border: 1px solid #555555 !important;
    }
    .widget-button:hover {
        background-color: #444444 !important;
    }
    /* Dark theme for Matplotlib outputs */
    .output_area, .output_subarea {
        background-color: #1e1e1e !important;
    }
</style>
"""))

# ==========================================
# 1. MAIN TITLE & INSTRUCTIONS
# ==========================================
main_title = widgets.HTML("""
    <div style="display: flex; align-items: center; border-bottom: 3px solid #d62728; padding-bottom: 15px; margin-bottom: 20px; background-color: #1e1e1e; color: white; padding: 15px; border-radius: 5px;">
        <span style="font-size: 48px; margin-right: 20px;">🦠</span>
        <h1 style="margin: 0; color: #ffffff; font-family: Arial, sans-serif;">
            The Resistance Curve: Tracking Community Prescribing Pressures and Pathogen Adaptability
        </h1>
    </div>
""")

instructions = widgets.HTML("""
    <div style="background-color: #2a2a2a; padding: 20px; border-radius: 8px; border-left: 5px solid #1f77b4; color: #dddddd; margin-bottom: 30px; font-family: Arial, sans-serif;">
        <h3 style="margin-top: 0; color: #1f77b4;">💡 How to Use This Dashboard</h3>
        <p>This dashboard tells a story in three parts. Follow these steps to explore the data:</p>
        <ol style="margin-bottom: 0;">
            <li><b>Read the Introduction:</b> Start at Section 1 to understand overall antibiotic prescribing trends and seasonal spikes (like winter).</li>
            <li><b>Explore the AMR Data (Section 2):</b> Use the <b>Dropdown menus</b> to select a specific Bacteria, Resistance Type, and Region. The chart will update automatically to show you the resistance trends for your selection. (Select "All Regions" to see a comparison of all UK regions).</li>
            <li><b>See the Interaction (Section 3):</b> Once you've found a specific bacteria you want to investigate, scroll down and click the <b>green "Update Interaction Chart" button</b>. This will generate a dual-axis graph showing how your selected bacteria's resistance correlates with total national antibiotic use over time.</li>
        </ol>
    </div>
""")

# ==========================================
# 2. SECTION 1: INTRODUCTION (ANTIBIOTIC USE)
# ==========================================
sec1_title = widgets.HTML("""
    <div style="color: #ffffff;">
        <h2>1. The Scale of Antibiotic Prescribing</h2>
        <p>Antibiotics are life-saving medicines, but the more they are used, the faster bacteria evolve to resist them. 
        Before we look at resistance, it is important to understand <i>how</i> and <i>when</i> these drugs are used across the UK.</p>
        <ul>
            <li><b>Primary vs. Secondary Care:</b> Most antibiotics are prescribed in Primary Care (GPs) compared to Secondary Care (Hospitals).</li>
            <li><b>Seasonality:</b> Use the heatmap to see the 'Winter Spike.' Antibiotic use naturally rises in colder months due to respiratory infections.</li>
        </ul>
        <p><i>Definitions: <b>Primary Care</b> = GPs and community dentists. <b>Secondary Care</b> = Hospitals. <b>Prescription Items</b> = Total number of prescriptions dispensed.</i></p>
    </div>
""")

# Load Use Data
df_use = pd.read_csv("data/processed/ukhsa_consumption_clean.csv")
df_use['date'] = pd.to_datetime(df_use['date'])
df_use['Year'] = df_use['date'].dt.year
df_use['Month'] = df_use['date'].dt.strftime('%b')

month_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
numeric_cols = df_use.select_dtypes(include='number').columns
use_col = [col for col in numeric_cols if col != 'Year'][0]

out_line = widgets.Output()
out_heat = widgets.Output()

plotly_dark_layout = dict(
    paper_bgcolor='#1e1e1e',
    plot_bgcolor='#1e1e1e',
    font=dict(color='#ffffff')
)

with out_line:
    if 'Sector' in df_use.columns:
        trend_data = df_use.groupby(['Year', 'Sector'])[use_col].sum().reset_index()
        fig_line = px.line(trend_data, x="Year", y=use_col, color="Sector", markers=True, title="Total Antibiotic Use Over Time", labels={use_col: "Total Prescriptions"})
    else:
        trend_data = df_use.groupby('Year')[use_col].sum().reset_index()
        fig_line = px.line(trend_data, x="Year", y=use_col, markers=True, title="Total Antibiotic Use Over Time", labels={use_col: "Total Prescriptions"})
    fig_line.update_layout(**plotly_dark_layout)
    fig_line.show()

with out_heat:
    pivot_df = df_use.pivot_table(index="Month", columns="Year", values=use_col, aggfunc="sum").reindex(month_order)
    fig_heat = px.imshow(pivot_df, labels=dict(x="Year", y="Month", color="Total Use"), title="Seasonality: When are antibiotics prescribed?", color_continuous_scale="Reds", aspect="auto")
    fig_heat.update_layout(**plotly_dark_layout)
    fig_heat.show()

sec1_plots = widgets.HBox([out_line, out_heat])


# ==========================================
# 3. SECTION 2: AMR EXPLORER (LIVE API)
# ==========================================
sec2_title = widgets.HTML("""
    <div style="color: #ffffff;">
        <h2>2. Explore Antimicrobial Resistance (AMR)</h2>
        <p>This data is fetched live from the UKHSA API. Antimicrobial Resistance (AMR) occurs when bacteria change in a way that reduces or eliminates the effectiveness of drugs.</p>
        <p><i>Definitions: <b>AMR</b> = Bacteria becoming resistant to drugs. <b>Resistance Rate</b> = The percentage of tested bacteria that were resistant to the chosen antibiotic.</i></p>
    </div>
""")

BASE = "https://api.ukhsa-dashboard.data.gov.uk"
 
def get_amr_metric(topic, metric, geography_type="UKHSA Region", page_size=365):
    geo_url = f"{BASE}/themes/infectious_disease/sub_themes/antimicrobial_resistance/topics/{topic}/geography_types/{geography_type}/geographies?format=json"
    geos = [g["name"] for g in requests.get(geo_url).json()]
    frames = []
    
    for geo in geos:
        url = (f"{BASE}/themes/infectious_disease/sub_themes/antimicrobial_resistance"
               f"/topics/{topic}/geography_types/{geography_type}/geographies/{geo}/metrics/{metric}"
               f"?format=json&page_size={page_size}")
        results = []
        while url:
            r = requests.get(url)
            r.raise_for_status()
            data = r.json()
            results.extend(data.get("results", []))
            url = data.get("next")
        if results:
            df = pd.DataFrame(results)
            df = df.rename(columns={"date":"date", "geography":"region", "metric_value":"resistance",
                                    "stratum":"resistance_type", "topic":"bacteria"})
            df = df[["date", "region", "resistance", "bacteria", "resistance_type"]]
            df["bacteria"] = df["bacteria"].str.replace("-", "_")
            frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

print("Downloading Live AMR Data from UKHSA API (This may take a moment)...")
CONFIG = [
    {"topic": "E-coli", "metric": "e-coli_testing_bacteraemiaPercentResistantRollingMonth", "label": "E_coli"},
    {"topic": "E-faecalis", "metric": "e-faecalis_testing_bacteraemiaPercentResistantRollingMonth", "label": "E_faecalis"},
    {"topic": "E-faecium", "metric": "e-faecium_testing_bacteraemiaPercentResistantRollingMonth", "label": "E_faecium"},
    {"topic": "K-pneumoniae", "metric": "k-pneumoniae_testing_bacteraemiaPercentResistantRollingMonth", "label": "K_pneumoniae"},
    {"topic": "P-aeruginosa", "metric": "p-aeruginosa_testing_bacteraemiaPercentResistantRollingMonth", "label": "P_aeruginosa"},
    {"topic": "S-pneumoniae", "metric": "s-pneumoniae_testing_bacteraemiaPercentResistantRollingMonth", "label": "S_pneumoniae"},
]

frames = []
for cfg in CONFIG:
    df = get_amr_metric(cfg["topic"], cfg["metric"])
    df["bacteria"] = cfg["label"]
    frames.append(df)

amr = pd.concat(frames, ignore_index=True)
amr["date"] = pd.to_datetime(amr["date"])

all_bacteria = amr["bacteria"].unique()
all_resistance = amr["resistance_type"].unique()
all_regions = amr["region"].unique()
all_dates = amr["date"].unique()

complete_index = pd.MultiIndex.from_product(
    [all_bacteria, all_resistance, all_regions, all_dates], 
    names=["bacteria", "resistance_type", "region", "date"]
)

amr = amr.set_index(["bacteria", "resistance_type", "region", "date"]) \
         .reindex(complete_index) \
         .reset_index()
amr = amr.sort_values(["bacteria", "resistance_type", "region", "date"])

csv_path = Path("data/processed/amr_all_bacteria.csv")
csv_path.parent.mkdir(parents=True, exist_ok=True)
amr.to_csv(csv_path, index=False)
print("Dataset successfully rebuilt!")

bacteria_dropdown = widgets.Dropdown(options=sorted(amr["bacteria"].unique()), description="Bacteria:", layout=widgets.Layout(width="220px"))
resistance_dropdown = widgets.Dropdown(options=[], description="Resistance:", layout=widgets.Layout(width="280px"))
region_dropdown = widgets.Dropdown(options=["All Regions"] + sorted(amr["region"].dropna().unique().tolist()), value="All Regions", description="Region:", layout=widgets.Layout(width="200px"))

out_amr_matplotlib = widgets.Output()

def update_amr_plot(change=None):
    with out_amr_matplotlib:
        clear_output(wait=True)
        bact = bacteria_dropdown.value
        rest = resistance_dropdown.value
        if not bact or not rest:
            print("Please make a selection.")
            return

        df = amr[(amr["bacteria"] == bact) & (amr["resistance_type"] == rest)]
        if region_dropdown.value != "All Regions":
            df = df[df["region"] == region_dropdown.value]
        df = df.dropna(subset=["resistance"])
        
        if df.empty:
            print("No data available for this selection.")
            return

        plt.style.use('dark_background')
        plt.figure(figsize=(13, 6))
        
        if region_dropdown.value == "All Regions":
            for region in sorted(df["region"].unique()):
                subset = df[df["region"] == region].sort_values("date")
                if not subset.empty:
                    plt.plot(subset["date"], subset["resistance"], label=region, linewidth=1.8)
            plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", title="Region", fontsize=8)
        else:
            plt.plot(df.sort_values("date")["date"], df.sort_values("date")["resistance"], color="cyan", linewidth=2.5)

        plt.title(f"{bact.replace('_', ' ')} – {rest}", fontsize=14)
        plt.ylabel("% Resistant")
        plt.grid(True, linestyle="--", alpha=0.2, color='white')
        plt.tight_layout()
        plt.show()

def update_resistance(change):
    bact = change["new"]
    available = sorted(amr[(amr["bacteria"] == bact) & (amr["resistance"].notna())]["resistance_type"].unique())
    resistance_dropdown.options = available
    if available:
        resistance_dropdown.value = available[0]
    update_amr_plot()

bacteria_dropdown.observe(update_resistance, names="value")
resistance_dropdown.observe(update_amr_plot, names="value")
region_dropdown.observe(update_amr_plot, names="value")

if len(bacteria_dropdown.options) > 0:
    bacteria_dropdown.value = bacteria_dropdown.options[0]

sec2_widgets = widgets.HBox([bacteria_dropdown, resistance_dropdown, region_dropdown])

# ==========================================
# 4. SECTION 3: INTERACTION
# ==========================================
sec3_title = widgets.HTML("""
    <div style="color: #ffffff;">
        <h2>3. The Interaction: Does Use Drive Resistance?</h2>
        <p>When antibiotics are used heavily, it creates evolutionary pressure, allowing resistant bacteria to survive. 
        This chart overlays <b>Total Antibiotic Use</b> (dashed blue line) against <b>Resistance %</b> (solid red line) for your selection above.</p>
    </div>
""")

out_interaction = widgets.Output()
btn_interaction = widgets.Button(description="Update Interaction Chart", button_style='success')

def draw_interaction(b=None):
    with out_interaction:
        clear_output(wait=True)
        try:
            bact = bacteria_dropdown.value
            rest = resistance_dropdown.value
            
            total_use = df_use.groupby("Year")[use_col].sum().reset_index()
            
            amr_trend = amr[(amr["bacteria"] == bact) & (amr["resistance_type"] == rest)]
                
            if region_dropdown.value != "All Regions":
                amr_trend = amr_trend[amr_trend["region"] == region_dropdown.value]
                
            amr_trend['Year'] = amr_trend['date'].dt.year
            amr_trend = amr_trend.groupby("Year")["resistance"].mean().reset_index()
            
            if amr_trend.empty:
                print("No AMR data to compare.")
                return

            fig = make_subplots(specs=[[{"secondary_y": True}]])
            
            fig.add_trace(go.Scatter(x=amr_trend["Year"], y=amr_trend["resistance"], name="Resistance %", mode="lines+markers", line=dict(color="#d62728", width=3)), secondary_y=False)
            fig.add_trace(go.Scatter(x=total_use["Year"], y=total_use[use_col], name="Total Use", mode="lines+markers", line=dict(color="#1f77b4", width=3, dash='dash')), secondary_y=True)
            
            fig.update_layout(
                title=f"Interaction: Total Use vs {bact.replace('_', ' ')} Resistance", 
                hovermode="x unified",
                **plotly_dark_layout
            )
            fig.update_yaxes(title_text="Resistance (%)", secondary_y=False, color="#d62728")
            fig.update_yaxes(title_text="Total Prescriptions", secondary_y=True, color="#1f77b4")
            
            fig.show()
        except Exception as e:
            print(f"Error generating interaction chart: {e}")

btn_interaction.on_click(draw_interaction)

# ==========================================
# 5. FOOTER (TAKEAWAY & REFERENCE) & ASSEMBLY
# ==========================================
footer_html = widgets.HTML("""
    <br><br>
    <div style="background-color: #331111; padding: 15px; border-left: 5px solid #d62728; border-radius: 5px; color: #ffffff;">
        <h3 style="margin-top:0; color: #d62728;">Key Takeaway</h3>
        <p style="margin-bottom:0;">Reducing unnecessary antibiotic prescribing—especially in winter and in primary care—is essential to slowing down AMR and ensuring these drugs work for future generations.</p>
    </div>
    <br>
    <div style="text-align: center; color: #888888; font-size: 12px; padding-top: 20px; border-top: 1px solid #444;">
        <p><b>Data Source:</b> UK Health Security Agency (UKHSA) AMR Dashboard API & UKHSA Antimicrobial Consumption Data.</p>
        <p>Data accessed via the UKHSA database. Processed for dashboard visualization.</p>
    </div>
""")

red_hr = widgets.HTML("<hr style='border: 1px solid #555555; margin: 30px 0;'>")

# ASSEMBLY (Added 'instructions' right below main_title)
dashboard = widgets.VBox([
    main_title,
    instructions,
    sec1_title,
    sec1_plots,
    red_hr,
    sec2_title,
    sec2_widgets,
    out_amr_matplotlib,
    red_hr,
    sec3_title,
    btn_interaction,
    out_interaction,
    red_hr,
    footer_html
], layout=widgets.Layout(padding="20px", background_color="#1e1e1e"))

display(dashboard)

# Trigger the initial plot
update_resistance({'new': bacteria_dropdown.value})

Dataset successfully rebuilt!
